# 02 · Détecteurs vs vérité terrain

Le moteur de l'Analyzer (`analyser.engine`) est appliqué au rejeu complet, avec les seuils par défaut
(`analyser.config.Thresholds`). Les alertes produites sont comparées aux épisodes de vérité terrain :

- **rappel** = épisodes détectés / épisodes ;
- **précision** = alertes qui correspondent à un épisode / alertes ;
- **délai** = ouverture de la première alerte − début de l'épisode.

Une alerte correspond à un épisode de la même machine et du même scénario si elle s'ouvre entre le début de l'épisode
et sa fin + 60 s.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent  # AWS/Analyser
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))  # repli si iCloud masque le .venv (voir README)

import matplotlib.pyplot as plt
import pandas as pd

import _viz as viz
from analyser import generator
from analyser.config import Thresholds
from analyser.evaluation import analyse, episodes_from_ground_truth, evaluate, load_replay, sweep
from analyser.replay_files import read_ground_truth

viz.apply_style()
pd.set_option("display.precision", 2)

REPLAY = ROOT / "data" / "replay" / "ref"
if not (REPLAY / "batches.jsonl").exists():  # jeu de référence : 10 machines x 8 h, seed 42
    generator.write_replay(generator.generate(devices=10, hours=8, seed=42), REPLAY)

fetched = load_replay(REPLAY / "batches.jsonl")
ground_truth = read_ground_truth(REPLAY / "ground_truth.csv")
episodes = episodes_from_ground_truth(ground_truth, data_end_ms=fetched.now_ms)
eps = viz.episodes_frame(episodes)
print(f"{len(fetched.readings):,} lectures, {len(ground_truth):,} événements de vérité terrain, {len(episodes)} épisodes")

In [ ]:
result = analyse(fetched, Thresholds())
alerts = viz.alerts_frame(result.alerts)
print(f"{len(alerts)} alertes")
pd.crosstab(alerts["scenario"], alerts["severity"])

## 1. Scores

Sur tous les épisodes, y compris ceux encore en cours à la fin du rejeu (les fuites mémoire qui n'ont pas encore abouti au crash) :

In [ ]:
viz.scores_frame(evaluate(result.alerts, episodes))

## 2. Chronologie des détections

Barres grises : épisodes de vérité terrain. Losanges : ouverture des alertes.

In [ ]:
viz.plot_timelines(eps, alerts, ["thermal_runaway", "memory_leak_fota", "out_of_memory", "silent_device"]);

## 3. Une machine jusqu'au crash mémoire

L'alerte `memory_leak_fota` (WARNING) s'ouvre après la deuxième baisse du heap consécutive à une mise à jour, plusieurs heures avant le crash `out_of_memory` (CRITICAL).

In [ ]:
oom = eps[eps["scenario"] == "out_of_memory"].sort_values("start").iloc[0]
leak = eps[(eps["scenario"] == "memory_leak_fota") & (eps["device_id"] == oom.device_id) & (eps["end"] == oom.end)].iloc[0]
dev = viz.readings_frame(r for r in fetched.readings if r.device_id == oom.device_id)
window = (leak.start_time - pd.Timedelta(minutes=10), oom.end_time + pd.Timedelta(minutes=10))
viz.plot_device(dev, eps[eps["device_id"] == oom.device_id], alerts[alerts["device_id"] == oom.device_id],
                title=f"{oom.device_id} — de la première fuite au crash", window=window);

## Limites

- Le simulateur produit des signaux très nets (bruit de ±0,3 °C et ±200 octets) : des scores parfaits ici ne garantissent
  pas les mêmes résultats sur de vraies machines. Le notebook 03 montre où les seuils commencent à se dégrader.
- `device_offline` (> 5 min sans données) n'a pas de vérité terrain : les silences simulés durent au plus 120 s.
  Il est couvert par les tests unitaires et visible dans Grafana quand le rejeu s'arrête.
- La fuite mémoire est signalée à la deuxième mise à jour suivie d'une baisse du heap : médiane ≈ 77 min après la première
  fuite, alors que le crash survient ≈ 4 h 30 après elle. Juste après un crash, la mise à jour qui suit le redémarrage n'a
  pas encore de référence de heap en `standby` : la détection attend alors une mise à jour de plus.